# texas housing market

cited sources: [Texas Real Estate Research Center - Texas A&M](https://trerc.tamu.edu/data/housing-activity/?data-State=Texas)

In [1]:
import os

os.getcwd()

'/home/rc/version-tab/notebooks'

In [4]:
import pandas as pd


os.chdir("../data")
df = pd.read_csv("housing-activity.csv")

# Preview
df.head()

,Market Name,Date,Sales,Sales YoY%,Dollar Volume,Dollar Volume YoY%,Average Price,Median Price,Active Listings EOM,Months Inventory
0,Texas,2024-01-01,330998,0.0069,139645371806,0.0285,421891,339000,115096,4.10
1,Texas,2023-01-01,328736,-0.1061,135773859573,-0.1069,413017,335000,88610,3.36
2,Texas,2022-01-01,367759,-0.1126,152020676217,-0.0178,413370,340000,65745,2.64
3,Texas,2021-01-01,414410,0.0628,154768720423,0.2551,373467,300000,48340,1.21
4,Texas,2020-01-01,389905,0.0940,123312182561,0.1835,316262,259950,78259,1.63


In [7]:
df.columns

Index(['Market Name', 'Date', 'Sales', 'Sales YoY%', 'Dollar Volume',
       'Dollar Volume YoY%', 'Average Price', 'Median Price',
       'Active Listings EOM', 'Months Inventory'],
      dtype='object')

In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 35 entries, 0 to 34
Data columns (total 10 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Market Name          35 non-null     object 
 1   Date                 35 non-null     object 
 2   Sales                35 non-null     int64  
 3   Sales YoY%           35 non-null     float64
 4   Dollar Volume        35 non-null     int64  
 5   Dollar Volume YoY%   35 non-null     float64
 6   Average Price        35 non-null     int64  
 7   Median Price         35 non-null     int64  
 8   Active Listings EOM  35 non-null     int64  
 9   Months Inventory     35 non-null     float64
dtypes: float64(3), int64(5), object(2)
memory usage: 2.9+ KB


In [10]:
# Feature engineering
df['Price Acceleration'] = df['Median Price'].diff().diff()
df['Demand Index'] = df['Sales'] / df['Active Listings EOM']
df['Inventory Pressure'] = df['Active Listings EOM'].pct_change()

In [11]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Dense, Dropout

regression_df = df.dropna(subset=['Median Price', 'Demand Index', 'Price Acceleration', 'Inventory Pressure'])

X_reg = regression_df[['Demand Index', 'Price Acceleration', 'Inventory Pressure']]
y_reg = regression_df['Median Price']

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X_reg, y_reg, test_size=0.2, random_state=42)

# Normalize
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Build model
regression_model = Sequential([
    Input(shape=(X_train_scaled.shape[1],)),
    Dense(128, activation='relu'),
    Dropout(0.3),
    Dense(64, activation='relu'),
    Dropout(0.2),
    Dense(1)
])
regression_model.compile(optimizer='adam', loss='mse', metrics=['mae'])

# Train
regression_model.fit(X_train_scaled, y_train, epochs=30, batch_size=16, validation_split=0.1)

2025-08-05 06:20:43.500232: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE4.1 SSE4.2 AVX AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Epoch 1/30
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 204ms/step - loss: 31238039552.0000 - mae: 160962.3125 - val_loss: 17433106432.0000 - val_mae: 127635.7266
Epoch 2/30
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 66ms/step - loss: 30536984576.0000 - mae: 159071.6875 - val_loss: 17433090048.0000 - val_mae: 127635.6562
Epoch 3/30
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 68ms/step - loss: 29481805824.0000 - mae: 156615.9688 - val_loss: 17433075712.0000 - val_mae: 127635.6016
Epoch 4/30
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 66ms/step - loss: 28495235072.0000 - mae: 152533.7656 - val_loss: 17433061376.0000 - val_mae: 127635.5391
Epoch 5/30
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 67ms/step - loss: 28772837376.0000 - mae: 154171.1719 - val_loss: 17433044992.0000 - val_mae: 127635.4922
Epoch 6/30
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 64ms/step - loss: 30685093888.0000 - mae: 158844.2188 - val_loss: 17433032704.0000 - val_mae: 127635.4375
Epoch 7/30
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 67ms/step - loss: 31893112832.0000 - mae: 161941.7031 - val_loss: 17433018368.0000 - val_m

In [12]:
# Rule-based labeling
def classify_market(row):
    if row['Sales YoY%'] > 10 and row['Months Inventory'] < 2.5:
        return 'hot'
    elif row['Sales YoY%'] < -5 or row['Months Inventory'] > 6:
        return 'cold'
    else:
        return 'stable'

df['MarketLabel'] = df.apply(classify_market, axis=1)

# Encode classes
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report
le = LabelEncoder()
y_clf = le.fit_transform(df['MarketLabel'])

X_clf = df[['Sales YoY%', 'Dollar Volume YoY%', 'Months Inventory']].dropna()
y_clf = y_clf[:len(X_clf)]

# Split
X_train_clf, X_test_clf, y_train_clf, y_test_clf = train_test_split(X_clf, y_clf, test_size=0.2, random_state=42)
X_train_clf = scaler.fit_transform(X_train_clf)
X_test_clf = scaler.transform(X_test_clf)

# Build classifier
classification_model = Sequential([
    Input(shape=(X_train_clf.shape[1],)),
    Dense(64, activation='relu'),
    Dropout(0.2),
    Dense(len(le.classes_), activation='softmax')
])
classification_model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# Train
history = classification_model.fit(X_train_clf, y_train_clf, epochs=20, batch_size=16, validation_split=0.1)

Epoch 1/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 174ms/step - accuracy: 0.5342 - loss: 0.7729 - val_accuracy: 0.3333 - val_loss: 0.8157
Epoch 2/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 68ms/step - accuracy: 0.6083 - loss: 0.6985 - val_accuracy: 0.3333 - val_loss: 0.8037
Epoch 3/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 64ms/step - accuracy: 0.6883 - loss: 0.6632 - val_accuracy: 0.3333 - val_loss: 0.7905
Epoch 4/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 61ms/step - accuracy: 0.8042 - loss: 0.6539 - val_accuracy: 0.3333 - val_loss: 0.7799
Epoch 5/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 64ms/step - accuracy: 0.6083 - loss: 0.6876 - val_accuracy: 0.3333 - val_loss: 0.7703
Epoch 6/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 65ms/step - accuracy: 0.7567 - loss: 0.6527 - val_accuracy: 0.3333 - val_loss: 0.7600
Epoch 7/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 84ms/step - accuracy: 0.7567 - loss: 0.5961 - val_accuracy: 0.3333 - val_loss: 0.7508
Epoch 8/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 63ms/step - accuracy: 0.7833 - loss: 0.6320 - val_accuracy: 0.3333 - val_loss: 0.7419

In [13]:
loss, mae = regression_model.evaluate(X_test_scaled, y_test)
print(f"Regression MAE: ${mae:,.2f}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step - loss: 18675812352.0000 - mae: 131358.7656
Regression MAE: $131,358.77


In [14]:
y_pred_probs = classification_model.predict(X_test_clf)
y_pred_labels = le.inverse_transform(y_pred_probs.argmax(axis=1))
print(classification_report(le.inverse_transform(y_test_clf), y_pred_labels))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step
              precision    recall  f1-score   support

        cold       1.00      0.67      0.80         3
      stable       0.80      1.00      0.89         4

    accuracy                           0.86         7
   macro avg       0.90      0.83      0.84         7
weighted avg       0.89      0.86      0.85         7

